# @before_model 与 @after_model

在每次模型调用前后执行，适合做内容过滤、消息预处理、响应审核等。

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

print("导入完成")

导入完成


## @before_model——模型调用前拦截

### 场景 1：消息预处理——限制对话长度

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_model
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

@before_model
def limit_context(state, runtime):
    """限制消息历史长度（当前版本 middleware 返回值不再合并到状态，此处仅作演示）"""
    messages = state.get("messages", [])
    print(f"当前消息数: {len(messages)}")
    return None

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, middleware=[limit_context], system_prompt="你是菜鸟教程 RUNOOB 的助手。")

# 注意：当前版本 before_model 返回 dict 不能直接修改消息状态
result = agent.invoke({"messages": [HumanMessage(content=x) for x in ["第一轮","第二轮","第三轮","第四轮","第五轮","第六轮","第七轮"]]})
print(f"消息数: {len(result['messages'])} （验证中间件正常执行）")


当前消息数: 7
消息数: 8 （验证中间件正常执行）


### 场景 2：内容过滤——屏蔽敏感词

In [6]:
from langchain.agents.middleware import before_model

SENSITIVE_WORDS = ["密码", "银行卡号", "身份证号"]

@before_model
def content_filter(state, runtime):
    """检查用户消息是否包含敏感词"""
    messages = state.get("messages", [])
    if not messages:
        return None
    content = str(messages[-1].content) if hasattr(messages[-1], 'content') else ""
    for word in SENSITIVE_WORDS:
        if word in content:
            print(f"[拦截] 检测到敏感词: {word}")
            return {
                "jump_to": "end",
                "messages": [HumanMessage(content=f"抱歉，不能处理包含「{word}」的请求。")]
            }
    return None

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, middleware=[content_filter], system_prompt="你是菜鸟教程 RUNOOB 的助手。")

result = agent.invoke({"messages": [HumanMessage(content="Python 怎么入门？")]})
print(f"正常: {result['messages'][-1].content[:60]}")

result = agent.invoke({"messages": [HumanMessage(content="我的银行卡号是多少？")]})
print(f"拦截: {result['messages'][-1].content}")

print("内容过滤中间件已创建")

正常: 学习 Python 入门，建议从以下几个方面入手：

1. **环境搭建**：下载安装 Python（推荐最新稳定版 3
[拦截] 检测到敏感词: 银行卡号
拦截: 抱歉，我无法知悉您的个人隐私信息，包括银行卡号。请妥善保管好自己的敏感数据，不要随意向他人透露。如有需要，建议您通过银行官方渠道查询。
内容过滤中间件已创建


## @after_model——模型调用后处理

### 场景 3：响应内容审核

In [7]:
from langchain.agents.middleware import after_model
from langchain.messages import AIMessage

@after_model
def response_audit(state, runtime):
    """审核模型回复"""
    messages = state.get("messages", [])
    if not messages:
        return None
    last_msg = messages[-1]
    content = str(last_msg.content) if hasattr(last_msg, 'content') else ""
    FORBIDDEN_TOPICS = ["政治", "暴力", "色情"]
    for topic in FORBIDDEN_TOPICS:
        if topic in content:
            return {"messages": [AIMessage(content="抱歉，我无法回答这个问题。请询问编程学习相关的内容。")]}
    return None

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, middleware=[response_audit], system_prompt="你是菜鸟教程 RUNOOB 的助手。")

result = agent.invoke({"messages": [HumanMessage(content="Python 有哪些学习资源？")]})
print(f"回复: {result['messages'][-1].content}")

print("响应审核中间件已创建")

回复: 推荐你从 **菜鸟教程（RUNOOB）** 的 [Python3 教程](https://www.runoob.com/python3/python3-tutorial.html) 入手，内容通俗易懂、实例丰富，非常适合零基础学习。

以下是其他高质量资源（按类型整理）：

**📘 官方文档**
- [Python 官方中文文档](https://docs.python.org/zh-cn/3/) – 权威、全面的语言参考。

**🌐 在线教程**
- [廖雪峰 Python 教程](https://www.liaoxuefeng.com/wiki/1016959663602400) – 深入浅出，适合有一定基础者。
- [Python 100 例](https://www.runoob.com/python/python-100-examples.html) – 菜鸟教程的练手项目。

**📚 书籍（经典）**
- 《Python编程：从入门到实践》 – 项目驱动，入门首选。
- 《流畅的Python》 – 进阶必读。

**🎥 视频课程（B站）**
- 黑马程序员 / 小甲鱼 / 莫烦Python 的系列视频，免费且系统。

建议你学完菜鸟教程的基础后，立刻动手做小项目（如爬虫、数据分析小工具），巩固知识。祝学习顺利！
响应审核中间件已创建


### 场景 4：自动追加免责声明

In [ ]:
from langchain.agents.middleware import after_model
from langchain.messages import AIMessage

@after_model
def append_disclaimer(state, runtime):
    """在模型回复后自动追加免责声明"""
    messages = state.get("messages", [])
    if not messages:
        return None
    last_msg = messages[-1]
    if (last_msg.type == "ai" and last_msg.content
            and not (hasattr(last_msg, 'tool_calls') and last_msg.tool_calls)):
        return {"messages": [AIMessage(content=last_msg.content + "\n\n---\n*以上内容由菜鸟教程 AI 助手生成，仅供参考。*")]}
    return None

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, middleware=[append_disclaimer], system_prompt="你是菜鸟教程的助手。")

result = agent.invoke({"messages": [HumanMessage(content="Python 容易学吗？")]})
print(result["messages"][-1].content)

Python 非常适合初学者入门，因为它语法简洁、接近自然语言，而且有丰富的学习资源和社区支持。相比于 C++ 或 Java，Python 的代码更易读，学习曲线相对平缓。不过，任何编程语言都需要通过实践来掌握，保持耐心、多写代码、多查阅资料（比如菜鸟教程上的例子），很快就能上手。

---
*以上内容由菜鸟教程 AI 助手生成，仅供参考。*
免责声明中间件已创建


## can_jump_to——流程跳转控制

在 `before_model` 和 `after_model` 中声明可以跳转的目标节点。

In [9]:
from langchain.agents.middleware import before_model

@before_model(can_jump_to=["end"])
def conditional_exit(state, runtime):
    """用户说再见时直接结束"""
    messages = state.get("messages", [])
    if not messages:
        return None
    content = str(messages[-1].content)
    if content.strip() in ["再见", "拜拜", "bye"]:
        return {"jump_to": "end", "messages": [{"role": "assistant", "content": "再见！期待下次为您服务。"}]}
    return None

print("条件退出中间件已定义")

条件退出中间件已定义


**术语：**

- **@before_model**：模型调用前钩子，可修改消息或跳转流程
- **@after_model**：模型调用后钩子，可审核输出或追加内容
- **can_jump_to**：声明中间件可跳转的目标节点
- **jump_to**：实际跳转指令，值可为 tools/model/end